In [38]:
import pandas as pd
from ucimlrepo import fetch_ucirepo

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV, KFold
from ucimlrepo import fetch_ucirepo
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.compose import TransformedTargetRegressor
import matplotlib.pyplot as plt
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    mean_absolute_percentage_error,
    r2_score
)

In [15]:
df = pd.read_excel("./RENDIMIENTO.xlsx")
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 752 entries, 0 to 751
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Nacional      752 non-null    int64  
 1   Grado         752 non-null    int64  
 2   Matricula     752 non-null    int64  
 3   Convocatoria  752 non-null    int64  
 4   EP2           752 non-null    float64
 5   EP1           752 non-null    float64
 6   EF            752 non-null    float64
 7   NF            752 non-null    float64
dtypes: float64(4), int64(4)
memory usage: 47.1 KB


In [24]:
df.groupby("Convocatoria").describe().T

df[df['NF'] == 7]

,Nacional,Grado,Matricula,Convocatoria,EP2,EP1,EF,NF
20,0,1,1,0,5.5,1.2,8.5,7.0
223,1,5,1,0,6.8,6.9,7.0,7.0
324,1,3,1,0,8.4,9.0,6.5,7.0
371,1,3,1,0,9.3,4.9,6.5,7.0


In [30]:
X = df.drop(columns=["EF","NF"])
y = df["EF"].values.ravel()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42
)

X.info()

<class 'pandas.DataFrame'>
RangeIndex: 752 entries, 0 to 751
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Nacional      752 non-null    int64  
 1   Grado         752 non-null    int64  
 2   Matricula     752 non-null    int64  
 3   Convocatoria  752 non-null    int64  
 4   EP2           752 non-null    float64
 5   EP1           752 non-null    float64
dtypes: float64(2), int64(4)
memory usage: 35.4 KB


In [39]:
pipe_ridge = Pipeline([
    ("scaler", StandardScaler()),
    ("regressor", Ridge(alpha=0.1, random_state=42))
])

pipe_lasso = Pipeline([
    ("scaler", StandardScaler()),
    ("regressor", Lasso(alpha=0.1, random_state=42))
])

pipe_elastic = Pipeline([
    ("scaler", StandardScaler()),
    ("regressor", ElasticNet(alpha=0.1, l1_ratio=0.6, random_state=42))
])

pipe_knn_reg = Pipeline([
    ("scaler", StandardScaler()),
    ("regressor", KNeighborsRegressor(n_neighbors=5, weights="distance", p=2))
])

pipe_tree = Pipeline([
    ("scaler", StandardScaler()),
    ("regressor", DecisionTreeRegressor(
    criterion="squared_error",
    max_depth=5,
    min_samples_split=10,
    min_samples_leaf=4,
    random_state=42
))])

pipe_random_forest = Pipeline([
    ("scaler", StandardScaler()),
    ("regressor", RandomForestRegressor(
    n_estimators=100,
    max_features=11,     # En regresión se suele evaluar un porcentaje mayor de características
    max_depth=8,
    random_state=42,
    n_jobs=-1
))])

pipe_modelo_gb  = Pipeline([
    ("scaler", StandardScaler()),
    ("regressor",   HistGradientBoostingRegressor(
    loss="squared_error",   # Alternativas: 'absolute_error', 'huber' (robusto a outliers)
    learning_rate=0.05,
    max_iter=100,
    max_depth=6,
    random_state=42
))])

pipe_svr_rbf = Pipeline([
    ("scaler", StandardScaler()),
    ("regressor", SVR(kernel="rbf", C=10.0, epsilon=0.1, gamma="scale"))
])

Pipelines = [
    ("Ridge", pipe_ridge),
    ("Lasso", pipe_lasso),
    ("ElasticNet", pipe_elastic),
    ("K-vecinos", pipe_knn_reg),
    ("Arbol",pipe_tree),
    ("Bosque aleatorio",pipe_random_forest),
    ("Modelo GB",pipe_modelo_gb),
    ("Modelo vector",pipe_svr_rbf)
]



for pipe_name, pipeline in Pipelines:

    pipeline.fit(X_train, y_train)

    y_pred = pipeline.predict(X_test)

    r2 = r2_score(y_test, y_pred)
    print("Resultados del Pipeline: ", pipe_name)
    print(f"R^2: {r2:.3f}")

Resultados del Pipeline:  Ridge
R^2: 0.481
Resultados del Pipeline:  Lasso
R^2: 0.485
Resultados del Pipeline:  ElasticNet
R^2: 0.486
Resultados del Pipeline:  K-vecinos
R^2: 0.523
Resultados del Pipeline:  Arbol
R^2: 0.459
Resultados del Pipeline:  Bosque aleatorio
R^2: 0.526
Resultados del Pipeline:  Modelo GB
R^2: 0.494
Resultados del Pipeline:  Modelo vector
R^2: 0.532
